In [11]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.10 2.0.2 2.3.3 1.6.1


In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3132

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c !="disease"]
X,y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.3, stratify=y, random_state =SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


# Part 1: Entropy and information gain by hand

Formulas:

$$H = -\sum_c p_c \log_2 p_c \qquad IG = H(\text{parent}) - \sum_k \frac{n_k}{n} H(\text{child}_k)$$

Here $p_c$ is the share of class $c$ in the node, and $n_k$ is the number of patients in child $k$.

## 1. Parent entropy

There are 12 patients: 6 have disease and 6 do not. So $p_1 = p_0 = 6/12 = 0.5$.

$$H = -\left(\tfrac{1}{2}\log_2\tfrac{1}{2} + \tfrac{1}{2}\log_2\tfrac{1}{2}\right) = -(0.5\cdot(-1) + 0.5\cdot(-1)) = \mathbf{1.000}\ \text{bit}$$

The classes are perfectly balanced, so the entropy is at its maximum.

## 2. Root split: gain of each feature

For each feature, the left child has feature = 1 and the right child has feature = 0.

**age_55_plus**
- feature = 1: 6 patients (5 disease, 1 healthy)
- feature = 0: 6 patients (1 disease, 5 healthy)
- $H_1 = H_0 = -(\tfrac56\log_2\tfrac56 + \tfrac16\log_2\tfrac16) = 0.2192 + 0.4308 = 0.6500$

$$IG = 1 - \left(\tfrac{6}{12}\cdot 0.6500 + \tfrac{6}{12}\cdot 0.6500\right) = 1 - 0.6500 = \mathbf{0.3500}$$

**exercise_angina**
- feature = 1: 4 patients (4 disease, 0 healthy), so $H_1 = 0$
- feature = 0: 8 patients (2 disease, 6 healthy)
- $H_0 = -(\tfrac14\log_2\tfrac14 + \tfrac34\log_2\tfrac34) = 0.5 + 0.3113 = 0.8113$

$$IG = 1 - \left(\tfrac{4}{12}\cdot 0 + \tfrac{8}{12}\cdot 0.8113\right) = 1 - 0.5409 = \mathbf{0.4591}$$

**male**
- feature = 1: 6 patients (3 disease, 3 healthy), so $H_1 = 1$
- feature = 0: 6 patients (3 disease, 3 healthy), so $H_0 = 1$

$$IG = 1 - \left(\tfrac{6}{12}\cdot 1 + \tfrac{6}{12}\cdot 1\right) = \mathbf{0.0000}$$

**high_bp**
- feature = 1: 6 patients (4 disease, 2 healthy)
- feature = 0: 6 patients (2 disease, 4 healthy)
- $H_1 = H_0 = -(\tfrac23\log_2\tfrac23 + \tfrac13\log_2\tfrac13) = 0.3900 + 0.5283 = 0.9183$

$$IG = 1 - 0.9183 = \mathbf{0.0817}$$

| feature | information gain at the root |
|---|---|
| age_55_plus | 0.3500 |
| **exercise_angina** | **0.4591** |
| male | 0.0000 |
| high_bp | 0.0817 |

**The root split is `exercise_angina`**, because it has the highest gain (0.4591).

## 3. One level down

The branch `exercise_angina = 1` is pure (4 disease, 0 healthy), so it becomes a leaf.

The branch `exercise_angina = 0` is not pure. It has 8 patients (p5 to p12): 2 with disease and 6 healthy. Its entropy is $H = 0.8113$.

Now we compute the gain of the three remaining features inside this branch.

**age_55_plus**
- feature = 1: p5, p7 (1 disease, 1 healthy), so $H = 1$
- feature = 0: p6, p8 to p12 (1 disease, 5 healthy), so $H = 0.6500$

$$IG = 0.8113 - \left(\tfrac28\cdot 1 + \tfrac68\cdot 0.6500\right) = 0.8113 - 0.7375 = \mathbf{0.0738}$$

**male**
- feature = 1: p7, p8, p9 (0 disease, 3 healthy), so $H = 0$
- feature = 0: p5, p6, p10, p11, p12 (2 disease, 3 healthy)
- $H = -(\tfrac25\log_2\tfrac25 + \tfrac35\log_2\tfrac35) = 0.5288 + 0.4422 = 0.9710$

$$IG = 0.8113 - \left(\tfrac38\cdot 0 + \tfrac58\cdot 0.9710\right) = 0.8113 - 0.6069 = \mathbf{0.2044}$$

**high_bp**
- feature = 1: p5, p10, p11 (1 disease, 2 healthy), so $H = 0.9183$
- feature = 0: p6, p7, p8, p9, p12 (1 disease, 4 healthy), so $H = 0.7219$

$$IG = 0.8113 - \left(\tfrac38\cdot 0.9183 + \tfrac58\cdot 0.7219\right) = 0.8113 - 0.7956 = \mathbf{0.0157}$$

| feature (inside `exercise_angina = 0`) | information gain |
|---|---|
| age_55_plus | 0.0738 |
| **male** | **0.2044** |
| high_bp | 0.0157 |

**The second split is `male`**, with gain 0.2044.

## Why is `male` useless at the root but best here?

At the root, both groups (male and female) had 3 disease and 3 healthy patients. So `male` alone did not separate the classes, and its gain was 0.

But the first split, `exercise_angina`, moved the three male patients with disease (p1, p2, p3) into the leaf on the other side. In the branch that is left, all three men (p7, p8, p9) are healthy. So `male` now separates the classes well.

This is an interaction between two features: how useful a feature is can depend on the splits above it. This is why a tree chooses a split again at every node, using only the data in that node.

In [13]:
# The 12 toy patients from the handout
toy = pd.DataFrame({
    "age_55_plus": [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "exercise_angina": [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "male": [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "high_bp": [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    "disease": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

def H(labels):
    """Entropy of a label array, in bits."""
    p = np.bincount(labels) / len(labels)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def gain(df, f):
    """Information gain of splitting df on the 0/1 feature f."""
    left = df[f] == 1
    return H(df["disease"].values) - (
        left.mean() * H(df["disease"][left].values)
        + (~left).mean() * H(df["disease"][~left].values)
    )

print("parent entropy:", H(toy["disease"].values))
for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    print("root", f, round(gain(toy, f), 4))
branch = toy[toy["exercise_angina"] == 0]
for f in ["age_55_plus", "male", "high_bp"]:
    print("branch", f, round(gain(branch, f), 4))

parent entropy: 1.0
root age_55_plus 0.35
root exercise_angina 0.4591
root male 0.0
root high_bp 0.0817
branch age_55_plus 0.0738
branch male 0.2044
branch high_bp 0.0157


# Part 2: A decision tree from scratch

I write the tree in NumPy. In this step I write three building blocks:

- `entropy(y)`: how mixed the labels in a node are.
- `information_gain(y, left)`: how much entropy a split removes.
- `best_split(X, y, min_samples_leaf)`: tries every feature and every threshold, and returns the split with the highest gain.

Rules I follow (from the task):
- Candidate thresholds are the midpoints between consecutive distinct sorted values of a feature.
- Rows with `x <= threshold` go left.
- A split is valid only if both children have at least `min_samples_leaf` rows.
- If two splits have the same gain, the lower feature index wins, then the lower threshold.

In [14]:
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    p = p[p > 0]           
    return float(-(p * np.log2(p)).sum()) + 0.0


def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    y = np.asarray(y)
    left = np.asarray(left, dtype=bool)
    n = len(y)
    n_left = left.sum()
    n_right = n - n_left
    return float(
        entropy(y) - (n_left / n) * entropy(y[left]) - (n_right / n) * entropy(y[~left])
    )


def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best = None
    for j in range(X.shape[1]):           
        values = np.unique(X[:, j])    
        for t in (values[:-1] + values[1:]) / 2:
            left = X[:, j] <= t
            n_left = left.sum()
            if n_left < min_samples_leaf or len(y) - n_left < min_samples_leaf:
                continue                         
            g = information_gain(y, left)
            if best is None or g > best[2] + 1e-12:
                best = (j, float(t), float(g))
    return best

In [15]:
y_toy = toy["disease"].values
print("parent entropy:", entropy(y_toy))
print("root gains:", [round(information_gain(y_toy, toy[f].values == 1), 4)
                      for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]])
branch = toy[toy["exercise_angina"] == 0]
print("branch gains:", [round(information_gain(branch["disease"].values, branch[f].values == 1), 4)
                        for f in ["age_55_plus", "male", "high_bp"]])

parent entropy: 1.0
root gains: [0.35, 0.4591, 0.0, 0.0817]
branch gains: [0.0738, 0.2044, 0.0157]


In [16]:
from sklearn.tree import DecisionTreeClassifier

ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED).fit(X_train, y_train)
print("sklearn:  ", feature_names[ref.tree_.feature[0]], ref.tree_.threshold[0])

feature, threshold, g = best_split(X_train, y_train)
print("my split: ", feature_names[feature], threshold, "gain =", round(g, 4))

sklearn:   thal 4.5
my split:  thal 4.5 gain = 0.2142


## Checks so far

**Do `entropy` and `information_gain` reproduce my hand values from Part 1?**
Yes. The parent entropy is 1.0, the root gains are 0.35, 0.4591, 0.0 and 0.0817, and the gains inside the `exercise_angina = 0` branch are 0.0738, 0.2044 and 0.0157. These are the same as my hand calculations.

**Does `best_split` choose the same split as scikit-learn?**
Yes. Both choose `thal` with threshold 4.5 on my training split. My gain for this split is 0.2142.

In [17]:
def majority_class(y):
    """Most frequent label; a tie goes to the smaller label."""
    labels, counts = np.unique(y, return_counts=True)
    return int(labels[np.argmax(counts)])


def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively. A node is a dict; a leaf has 'leaf': True and a 'prediction'."""
    leaf = {"leaf": True, "prediction": majority_class(y), "n": int(len(y))}
    if len(np.unique(y)) == 1:                         
        return leaf
    if max_depth is not None and depth == max_depth: 
        return leaf
    split = best_split(X, y, min_samples_leaf)
    if split is None:                                  
        return leaf
    j, t, g = split
    left = X[:, j] <= t
    return {
        "leaf": False, "feature": j, "threshold": t, "gain": g, "n": int(len(y)),
        "left": build_tree(X[left], y[left], max_depth, min_samples_leaf, depth + 1),
        "right": build_tree(X[~left], y[~left], max_depth, min_samples_leaf, depth + 1),
    }


def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    out = np.empty(len(X), dtype=int)

    def fill(node, idx):
        if node["leaf"]:
            out[idx] = node["prediction"]
            return
        go_left = X[idx, node["feature"]] <= node["threshold"]
        fill(node["left"], idx[go_left])
        fill(node["right"], idx[~go_left])

    fill(tree, np.arange(len(X)))
    return out

In [18]:
rows = []
for d in [1, 2, 3, 4, 5, None]:
    mine = build_tree(X_train, y_train, max_depth=d)
    ref = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED).fit(X_train, y_train)
    different = int((predict_tree(mine, X_test) != ref.predict(X_test)).sum())
    rows.append({
        "max_depth": "unlimited" if d is None else d,
        "rows that differ (of 90)": different,
        "same label": f"{1 - different / len(X_test):.1%}",
    })
pd.DataFrame(rows)

,max_depth,rows that differ (of 90),same label
0,1,0,100.0%
1,2,0,100.0%
2,3,0,100.0%
3,4,0,100.0%
4,5,1,98.9%
5,unlimited,8,91.1%


## Answers to the questions

| Question | Answer |
|---|---|
| Do `entropy` and `information_gain` reproduce the hand values of Part 1? | Yes: entropy 1.0, root gains 0.35, 0.4591, 0.0, 0.0817, branch gains 0.0738, 0.2044, 0.0157. |
| Does `best_split` choose the same split as scikit-learn at `max_depth=1`? | Yes: `thal`, threshold 4.5 (gain 0.2142). |
| Same label as scikit-learn on the test rows? | 100% for depths 1 to 4, 98.9% (1 row differs) for depth 5, 91.1% (8 rows differ) for unlimited depth. |

Below 100% at depth 5 and unlimited depth, the cause is ties. Sometimes two splits have exactly the same gain. My code picks the lower feature index, while scikit-learn looks at the features in a random order and takes the first one it finds. For example, at depth 5 one node with 3 rows has gain 0.9183 for both `cp` (mine) and `restecg` (scikit-learn). The trees then differ in a few small leaves.